<a href="https://colab.research.google.com/github/anupprogrammer/AccordionView/blob/master/notebooks/Vanna_May2026_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install the Package
Here we're installing it directly from GitHub while it's in development.

In [1]:
!pip install 'vanna[flask,anthropic]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 763.1/763.1 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.6/486.6 kB 25.0 MB/s eta 0:00:00


# Download a Sample Database

In [2]:
import httpx

with open("Chinook.sqlite", "wb") as f:
    with httpx.stream("GET", "https://vanna.ai/Chinook.sqlite") as response:
        for chunk in response.iter_bytes():
            f.write(chunk)

# Imports

In [3]:
from vanna import Agent, AgentConfig
from vanna.servers.fastapi import VannaFastAPIServer
from vanna.core.registry import ToolRegistry
from vanna.core.user import UserResolver, User, RequestContext
from vanna.integrations.anthropic import AnthropicLlmService
from vanna.tools import RunSqlTool, VisualizeDataTool
from vanna.integrations.sqlite import SqliteRunner
from vanna.tools.agent_memory import SaveQuestionToolArgsTool, SearchSavedCorrectToolUsesTool
from vanna.integrations.local.agent_memory import DemoAgentMemory
from vanna.capabilities.sql_runner import RunSqlToolArgs
from vanna.tools.visualize_data import VisualizeDataArgs

# Define your User Authentication
Here we're going to say that if you're logged in as `admin@example.com` then you're in the `admin` group, otherwise you're in the `user` group

In [4]:
class SimpleUserResolver(UserResolver):
    async def resolve_user(self, request_context: RequestContext) -> User:
        # In production, validate cookies/JWTs here
        user_email = request_context.get_cookie('vanna_email')
        if not user_email:
            raise ValueError("Missing 'vanna_email' cookie for user identification")

        print(f"Resolving user for email: {user_email}")

        if user_email == "admin@example.com":
            return User(id="admin1", email=user_email, group_memberships=['admin'])

        return User(id="user1", email=user_email, group_memberships=['user'])

# Define the Tools and Access Control

In [5]:
tools = ToolRegistry()
tools.register_local_tool(RunSqlTool(sql_runner=SqliteRunner(database_path="./Chinook.sqlite")), access_groups=['admin', 'user'])
tools.register_local_tool(VisualizeDataTool(), access_groups=['admin', 'user'])
agent_memory = DemoAgentMemory(max_items=1000)
tools.register_local_tool(SaveQuestionToolArgsTool(), access_groups=['admin'])
tools.register_local_tool(SearchSavedCorrectToolUsesTool(), access_groups=['admin', 'user'])

In [6]:
# Set up LLM
llm = AnthropicLlmService(model="claude-sonnet-4-5", api_key="sk-ant-...")

# Create agent with your options
agent = Agent(
    llm_service=llm,
    tool_registry=tools,
    user_resolver=SimpleUserResolver(),
    config=AgentConfig(),
    agent_memory=agent_memory
)

# 4. Create and run server
server = VannaFastAPIServer(agent)
server.run()

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

Your app is running at:
https://8000-m-s-kkb-usc1b2-xuhdmv247mku-b.us-central1-2.prod.colab.dev


INFO:     Started server process [3954]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:53668 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:48448 - "POST /api/vanna/v2/chat_sse HTTP/1.1" 200 OK


ERROR:vanna.core.agent.agent:Error in send_message (conversation_id=1778786810466-z2xe02l0f): Missing 'vanna_email' cookie for user identification
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/vanna/core/agent/agent.py", line 162, in send_message
    async for component in self._send_message(
  File "/usr/local/lib/python3.12/dist-packages/vanna/core/agent/agent.py", line 257, in _send_message
    user = await self.user_resolver.resolve_user(request_context)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_3954/1146226105.py", line 6, in resolve_user
    raise ValueError("Missing 'vanna_email' cookie for user identification")
ValueError: Missing 'vanna_email' cookie for user identification
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/vanna/core/agent/agent.py", line 162, in send_message
    async for component in self._send_message(
  File "/usr/local/lib/python3.12/dist-p

INFO:     127.0.0.1:48462 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:33010 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:33018 - "POST /api/vanna/v2/chat_sse HTTP/1.1" 200 OK
Resolving user for email: admin@example.com
INFO:     127.0.0.1:42560 - "POST /api/vanna/v2/chat_sse HTTP/1.1" 200 OK
Resolving user for email: admin@example.com


ERROR:vanna.core.agent.agent:Error in send_message (conversation_id=1778786864461-b6n5adr44): Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011Cb31i39sf5d2yiaa6MECc'}
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/vanna/core/agent/agent.py", line 162, in send_message
    async for component in self._send_message(
  File "/usr/local/lib/python3.12/dist-packages/vanna/core/agent/agent.py", line 653, in _send_message
    response = await self._handle_streaming_response(request)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/vanna/core/agent/agent.py", line 1356, in _handle_streaming_response
    async for chunk in self.llm_service.stream_request(request):
  File "/usr/local/lib/python3.12/dist-packages/vanna/integrations/anthropic/llm.py", line 105, in stream_request
    with self._client.messages.stream(**pa

KeyboardInterrupt: 